# NTHU COPILOT Course Planning Agent

本 notebook 展示清大電機系 EE112 入學年度的課程規劃助理。畢業規則、課表衝堂與課程推薦皆由 deterministic tools 執行；Agent 只負責依照工具結果整理說明。

## Demo 0: Screenshot OCR Cache Setup

**Step 1.** 請先在 `Python (OCR)` kernel 執行 `ocr_preprocess_demo.py`，它會讀取 `data/course_screenshot.png` 並產生 `data/course_screenshot_ocr.txt`。

**Step 2.** 回到本 notebook（`Python (HW2)` kernel），下方程式會讀取 OCR cache，建立暫定修課紀錄，並停下來讓你確認或修正。

按 Enter 才會正式儲存 OCR confirmed record，Demo 1 之後才會使用這份資料。


In [1]:
from pathlib import Path
import importlib
import pandas as pd
from IPython.display import display, Markdown

import course_data_loader
import graduation_rules
import course_review_searcher
import course_recommender
import course_agent
import ocr_screenshot_parser

# Reload local modules so notebook reruns pick up edits without needing a full kernel restart.
importlib.reload(course_data_loader)
importlib.reload(graduation_rules)
importlib.reload(course_review_searcher)
importlib.reload(course_recommender)
importlib.reload(course_agent)
importlib.reload(ocr_screenshot_parser)

from course_data_loader import load_target_courses
from graduation_rules import load_rules, check_graduation_progress
from course_recommender import recommend_courses, update_plan
from course_agent import CoursePlanningAgent, explain_graduation_result
from evaluation_report import run_evaluation_report
from calendar_exporter import export_schedule_to_ics
from ocr_screenshot_parser import parse_course_screenshot_from_cache, interactive_confirm_ocr_record

current_semester = "11410"
target_semester = "11420"
admission_year = "112"
department = "EE"
planning_mode = "assume_in_progress_pass"

DATA_DIR = Path("data")
target_path = DATA_DIR / "114_2_course_data.xlsx"
rules_path = DATA_DIR / "rules" / "EE_112_rules.json"
screenshot_path = DATA_DIR / "course_screenshot.png"
ocr_confirmed_student_path = DATA_DIR / "ocr_confirmed_student_courses.csv"
historical_course_paths = [
    DATA_DIR / "111-113 _course_data.xlsx",
    DATA_DIR / "114_1_course_data.xlsx",
]

target_df = load_target_courses(target_path)
rules = load_rules(rules_path)

print("Course database loaded successfully.")
print(f"- 114 第二學期 course database: {len(target_df)} courses")

ocr_result = parse_course_screenshot_from_cache(
    image_path=str(screenshot_path),
    historical_course_paths=[str(path) for path in historical_course_paths],
    target_path=str(target_path),
    force_reparse=True,
)

if ocr_result.get("warnings"):
    display(Markdown("\n".join(f"- {warning}" for warning in ocr_result["warnings"])))

confirmed_ocr = interactive_confirm_ocr_record(
    ocr_result,
    target_path=str(target_path),
    output_path=str(ocr_confirmed_student_path),
)

if not confirmed_ocr.get("confirmed"):
    raise RuntimeError("OCR 修課紀錄尚未確認，請先完成 Demo 0。")

student_path = Path(confirmed_ocr["output_path"])
student_df = pd.DataFrame(confirmed_ocr["confirmed_courses"])

display(Markdown(f"已確認修課紀錄，Demo 1 會使用：`{student_path}`"))


Course database loaded successfully.
- 114 第二學期 course database: 2895 courses


### OCR 修課紀錄確認

系統已根據截圖與歷史課程資料辨識出以下修課紀錄，請確認。確認後才會送進 agent。

,term,academic_year,semester,normalized_course_code,raw_course_code,course_name_zh,course_name_en,credits,status,grade,needs_user_confirmation,match_confidence,match_source
0,11210,112,10,EE2310,EE231001,計算機程式設計,Introduction to Programming,3.0,completed,pass,False,1.0,ocr_exact
1,11210,112,10,EECS1010,EECS101001,邏輯設計,Logic Design,3.0,completed,pass,False,1.0,ocr_exact
2,11210,112,10,LANG1010,LANG101017,中高級英文一,Upper-Intermediate English I,2.0,completed,pass,False,1.0,ocr_exact
3,11210,112,10,MATH1010,MATH101006,微積分一(數學系),Calculus (I),4.0,completed,pass,False,1.0,ocr_exact
4,11210,112,10,PE1110,PE111017,大一體育,Physical Education,0.0,completed,pass,False,1.0,ocr_exact
5,11210,112,10,PHYS1010,PHYS101009,普通物理實驗一,General Physics Laboratory (I),1.0,completed,pass,False,1.0,ocr_exact
6,11210,112,10,PHYS1133,PHYS113305,普通物理Ｂ一,General Physics B (I),3.0,completed,pass,False,1.0,ocr_exact
7,11220,112,20,EE2230,EE223002,邏輯設計實驗,Logic Design Laboratory,2.0,completed,pass,False,1.0,ocr_exact
8,11220,112,20,EECS2030,EECS203002,常微分方程,Ordinary Differential Equations,3.0,completed,pass,False,1.0,ocr_exact
9,11220,112,20,GE1198,GE119800,十八世紀英國的藝術與社會,Introduction to Eighteenth-century English Art...,2.0,completed,pass,False,1.0,ocr_exact


如果都正確，請直接按 **Enter** 繼續。  
如果有漏課，請輸入例如：`漏了機率`。  
如果要修正狀態/成績，請輸入例如：`我要修正電子學二的狀態` 或 `我要修正電子學二的成績`。

請輸入修正內容（直接 Enter 確認）： 


已確認 OCR 修課紀錄，並儲存為 `data/ocr_confirmed_student_courses.csv`。

,term,academic_year,semester,normalized_course_code,raw_course_code,course_name_zh,course_name_en,credits,status,grade,needs_user_confirmation,match_confidence,match_source
0,11210,112,10,EE2310,EE231001,計算機程式設計,Introduction to Programming,3.0,completed,pass,False,1.0,ocr_exact
1,11210,112,10,EECS1010,EECS101001,邏輯設計,Logic Design,3.0,completed,pass,False,1.0,ocr_exact
2,11210,112,10,LANG1010,LANG101017,中高級英文一,Upper-Intermediate English I,2.0,completed,pass,False,1.0,ocr_exact
3,11210,112,10,MATH1010,MATH101006,微積分一(數學系),Calculus (I),4.0,completed,pass,False,1.0,ocr_exact
4,11210,112,10,PE1110,PE111017,大一體育,Physical Education,0.0,completed,pass,False,1.0,ocr_exact
5,11210,112,10,PHYS1010,PHYS101009,普通物理實驗一,General Physics Laboratory (I),1.0,completed,pass,False,1.0,ocr_exact
6,11210,112,10,PHYS1133,PHYS113305,普通物理Ｂ一,General Physics B (I),3.0,completed,pass,False,1.0,ocr_exact
7,11220,112,20,EE2230,EE223002,邏輯設計實驗,Logic Design Laboratory,2.0,completed,pass,False,1.0,ocr_exact
8,11220,112,20,EECS2030,EECS203002,常微分方程,Ordinary Differential Equations,3.0,completed,pass,False,1.0,ocr_exact
9,11220,112,20,GE1198,GE119800,十八世紀英國的藝術與社會,Introduction to Eighteenth-century English Art...,2.0,completed,pass,False,1.0,ocr_exact


已確認修課紀錄，Demo 1 會使用：`data/ocr_confirmed_student_courses.csv`

## Demo 1: Graduation Progress Check

情境：EE112 入學學生想確認目前完成與修課中的課程，還缺哪些畢業需求。

本 demo 使用 Demo 0 已確認的 OCR 結構化資料 `student_df` 作為輸入。


In [2]:
graduation_input_source = "OCR confirmed student_df"

display(Markdown(f"**Graduation input source:** `{graduation_input_source}`"))
graduation_result = check_graduation_progress(student_df, rules, planning_mode=True)

print("正式採計課程：")
display(pd.DataFrame(graduation_result["completed_courses_official"]))

print("規劃模式下暫時計入的修課中課程：")
display(pd.DataFrame(graduation_result["in_progress_courses_counted_in_planning"]))

print("尚缺必修：")
display(pd.DataFrame(graduation_result["missing_required"]))

print("替代必修狀態：")
display(pd.DataFrame.from_dict(graduation_result["alternative_requirements_status"], orient="index"))


**Graduation input source:** `OCR confirmed student_df`

正式採計課程：


,code,raw_course_code,course_name_zh,course_name_en,credits,term,status,grade
0,EE2310,EE231001,計算機程式設計,Introduction to Programming,3.0,11210,completed,pass
1,EECS1010,EECS101001,邏輯設計,Logic Design,3.0,11210,completed,pass
2,LANG1010,LANG101017,中高級英文一,Upper-Intermediate English I,2.0,11210,completed,pass
3,MATH1010,MATH101006,微積分一(數學系),Calculus (I),4.0,11210,completed,pass
4,PE1110,PE111017,大一體育,Physical Education,0.0,11210,completed,pass
5,PHYS1010,PHYS101009,普通物理實驗一,General Physics Laboratory (I),1.0,11210,completed,pass
6,PHYS1133,PHYS113305,普通物理Ｂ一,General Physics B (I),3.0,11210,completed,pass
7,EE2230,EE223002,邏輯設計實驗,Logic Design Laboratory,2.0,11220,completed,pass
8,EECS2030,EECS203002,常微分方程,Ordinary Differential Equations,3.0,11220,completed,pass
9,GE1198,GE119800,十八世紀英國的藝術與社會,Introduction to Eighteenth-century English Art...,2.0,11220,completed,pass


規劃模式下暫時計入的修課中課程：


,code,raw_course_code,course_name_zh,course_name_en,credits,term,status,grade
0,EE3350,EE335000,固態電子元件導論,Introduction to Solid-State Electronic Devices,3.0,11410,in_progress,
1,GEC1402,GEC140201,社會文化分析,Social and cultural Analysis,2.0,11410,in_progress,
2,EE2245,EE224501,電子電路實驗,Microelectronics Labs.,2.0,11410,in_progress,
3,EE2260,EE226001,電子學二,Electronics II,3.0,11410,in_progress,


尚缺必修：


,code,name_zh,credits
0,EE2020,偏微分方程與複變函數,3
1,EE2140,電磁學,3
2,EE3900,實作專題一,1
3,EE3910,實作專題二,2


替代必修狀態：


,label,name_zh,satisfied,status,selected_code,options,missing_options
linear_algebra,Linear Algebra,線性代數,False,missing,,"[{'code': 'EE2030', 'name_zh': '線性代數', 'credit...","[{'code': 'EE2030', 'name_zh': '線性代數', 'credit..."
probability,Probability,機率,False,missing,,"[{'code': 'EE3060', 'name_zh': '機率', 'credits'...","[{'code': 'EE3060', 'name_zh': '機率', 'credits'..."


In [3]:
display(Markdown(explain_graduation_result(graduation_result)))


### 畢業進度檢查

已依照 **EE112 入學年度規則** 進行檢查。

| 項目 | 結果 |
|---|---|
| 已完成並正式採計的必修 | `MATH1010` 微積分一(數學系)<br>`MATH1020` 微積分二(數學系)<br>`PHYS1133` 普通物理Ｂ一<br>`PHYS1143` 普通物理Ｂ二<br>`PHYS1010` 普通物理實驗一<br>`EECS1010` 邏輯設計<br>`EE2310` 計算機程式設計<br>`EECS2030` 常微分方程<br>`EE2210` 電路學<br>`EE2255` 電子學<br>`EECS2020` 訊號與系統 |
| 規劃模式下暫時計入的修課中必修 | `EE2245` 電子電路實驗 |
| 尚缺必修 | `EE2020` 偏微分方程與複變函數<br>`EE2140` 電磁學<br>`EE3900` 實作專題一<br>`EE3910` 實作專題二 |
| 尚缺替代必修群組 | Linear Algebra, Probability |
| 必選實驗進度 | 2/6 學分，1/3 門 |
| 學分摘要 | 正式完成 47 學分；若修課中皆通過，規劃採計 57 學分 |

#### 注意事項
- EE112 規則要求 `EE2255 電子學`；`EE2250 電子學一` 與 `EE2260 電子學二` 可能和課程調整有關，但 NTHU COPILOT 不能自動認定可抵免，請向系辦確認。
- 修課中的課只是在規劃模式下假設會通過；如果未通過，畢業進度與推薦課表需要重新計算。
- NTHU COPILOT 不是正式畢業審查，最後仍須以系辦確認為準。

## Demo 2: Interactive Course Planning Loop

用同一個 `CoursePlanningAgent` 物件進行互動式排課。你可以連續輸入需求，例如加課、刪課、查指定時段候選課、查老師評價，直到輸入 `done` 或 `我決定好了` 確認課表。

In [3]:
agent = CoursePlanningAgent(
    student_path=student_path,
    target_path=target_path,
    rules_path=rules_path,
    use_llm=False,
    use_llm_intent=True,
)
agent.load()

print("NTHU COPILOT Course Planning started.")
print("請先輸入你的排課需求，例如：")
print("幫我排 114 第二學期課表，16-22 學分，不要星期五，不要太硬，可以參考網路評價")
print("之後可以繼續輸入：我不想修某門課 / 某門課評價怎麼樣 / 幫我根據網路評價重新排課 / 少一點學分")
print("如果決定好了，請輸入：我決定好了 / final / done / 確定。")

final_plan = None
confirmation_keywords = ["我決定好了", "確定了", "就這樣", "最終課表"]

while True:
    user_message = input("\n請輸入你的需求或修改意見：").strip()

    if user_message.lower() in ["final", "done", "confirm", "ok"] or any(
        keyword in user_message for keyword in confirmation_keywords
    ):
        result = agent.chat(user_message)
        final_plan = result
        display(Markdown(result.get("agent_explanation", "目前還沒有產生課表。")))

        confirmed_schedule = agent.last_recommendation or final_plan or {}
        final_courses = confirmed_schedule.get("recommended_courses", []) if isinstance(confirmed_schedule, dict) else []
        if final_courses:
            calendar_result = export_schedule_to_ics(final_courses, output_path="final_schedule.ics")
            display(Markdown(
                f"### Calendar Export\n"
                f"已自動匯出行事曆檔：`{calendar_result['ics_path']}`"
            ))
            display(Markdown(calendar_result["preview_markdown"]))
        else:
            display(Markdown("### Calendar Export\n目前沒有可匯出的課程。請先產生並確認課表。"))
        break

    previous_schedule = agent.last_recommendation
    result = agent.chat(user_message)

    # Demo 2 is the planning loop: if a natural preference sentence did not produce a schedule,
    # automatically reinterpret it as a planning request instead of showing only help text.
    if agent.last_recommendation is previous_schedule and result.get("intent") in ["help", "need_course_name_for_review"]:
        result = agent.chat(f"幫我排 114 第二學期課表，{user_message}")

    final_plan = agent.last_recommendation or result
    display(Markdown(result.get("agent_explanation", "No explanation returned.")))

# Keep the latest schedule available after the loop.
recommendation = agent.last_recommendation
preferences = dict(agent.last_preferences)

NTHU COPILOT Course Planning started.
請先輸入你的排課需求，例如：
幫我排 114 第二學期課表，16-22 學分，不要星期五，不要太硬，可以參考網路評價
之後可以繼續輸入：我不想修某門課 / 某門課評價怎麼樣 / 幫我根據網路評價重新排課 / 少一點學分
如果決定好了，請輸入：我決定好了 / final / done / 確定。



請輸入你的需求或修改意見： 我要20學分


### Course Planning

**產生 114 第二學期課表建議**
### 聊天記錄

1. `我要20學分`

### 114 第二學期課表建議

#### 推薦課程與理由
| 課號 | 課名 | 授課老師 | 學分 | 時間 | 為什麼建議修 | 評價參考 |
|---|---|---|---:|---|---|---|
| EE2020 | 偏微分方程與複變函數 | 陳國平 | 3 | `T3T4R3R4` | 補 EE112 必修缺口。 |  |
| EE2140 | 電磁學 | 黃承彬 | 3 | `M3M4W2` | 補 EE112 必修缺口。 |  |
| EE3900 | 實作專題一 | 指導教授 | 1 | `S8` | 補 EE112 必修缺口。 |  |
| EE3060 | 機率 | 謝欣霖 | 3 | `T5T6R5R6` | 補機率替代必修需求。 |  |
| EE2405 | 嵌入式系統與實驗 | 劉靖家 | 3 | `W7W8W9WaWb` | 依照初始 EE-first 排課策略，在未排除實驗課時加入 1 門電機實驗課。 |  |
| GE1024 | 產業創新實作 | 商雅婷,李佳玫,蔡協亨 | 2 | `TaTb` | 用 GE/GEC 通識課補其餘選修與學分。 |  |
| GE1026 | 紀錄片賞析與創作 | 蕭菊貞 | 3 | `WnW5W6` | 用 GE/GEC 通識課補其餘選修與學分。 |  |
| GE1073 | 智慧財產權 | 楊智傑 | 2 | `R1R2` | 用 GE/GEC 通識課補其餘選修與學分。 |  |
| PE1120 | 重量訓練 | 李柏均 | 0 | `M5M6` | 體育課可放進課表平衡生活節奏，但這門是 0 學分，不計入畢業總學分。 |  |

#### 每週課表
<table style="width:100%;table-layout:fixed;border-collapse:collapse;font-size:13px;line-height:1.25;">
<colgroup>
<col style="width:6%;">
<col style="width:13%;">
<col style="width:11.57%;">
<col style="width:11.57%;">
<col style="width:11.57%;">
<col style="width:11.57%;">
<col style="width:11.57%;">
<col style="width:11.57%;">
<col style="width:11.57%;">
</colgroup>
<thead>
<tr>
<th style="border-bottom:1px solid #bbb;padding:6px 4px;text-align:center;vertical-align:middle;background:#fff;font-weight:700;">節次</th>
<th style="border-bottom:1px solid #bbb;padding:6px 4px;text-align:center;vertical-align:middle;background:#fff;font-weight:700;">時間</th>
<th style="border-bottom:1px solid #bbb;padding:6px 4px;text-align:center;vertical-align:middle;background:#fff;font-weight:700;">M<br>一</th>
<th style="border-bottom:1px solid #bbb;padding:6px 4px;text-align:center;vertical-align:middle;background:#fff;font-weight:700;">T<br>二</th>
<th style="border-bottom:1px solid #bbb;padding:6px 4px;text-align:center;vertical-align:middle;background:#fff;font-weight:700;">W<br>三</th>
<th style="border-bottom:1px solid #bbb;padding:6px 4px;text-align:center;vertical-align:middle;background:#fff;font-weight:700;">R<br>四</th>
<th style="border-bottom:1px solid #bbb;padding:6px 4px;text-align:center;vertical-align:middle;background:#fff;font-weight:700;">F<br>五</th>
<th style="border-bottom:1px solid #bbb;padding:6px 4px;text-align:center;vertical-align:middle;background:#fff;font-weight:700;">S<br>六</th>
<th style="border-bottom:1px solid #bbb;padding:6px 4px;text-align:center;vertical-align:middle;background:#fff;font-weight:700;">U<br>日</th>
</tr>
</thead>
<tbody>
<tr style="background:#fff;">
<td style="border-bottom:1px solid #eee;padding:6px 4px;vertical-align:top;text-align:center;white-space:nowrap;font-weight:600;">1</td>
<td style="border-bottom:1px solid #eee;padding:6px 4px;vertical-align:top;text-align:center;white-space:nowrap;">08:00-08:50</td>
<td style="border-bottom:1px solid #eee;padding:6px 4px;vertical-align:top;text-align:left;overflow-wrap:anywhere;word-break:break-word;"></td>
<td style="border-bottom:1px solid #eee;padding:6px 4px;vertical-align:top;text-align:left;overflow-wrap:anywhere;word-break:break-word;"></td>
<td style="border-bottom:1px solid #eee;padding:6px 4px;vertical-align:top;text-align:left;overflow-wrap:anywhere;word-break:break-word;"></td>
<td style="border-bottom:1px solid #eee;padding:6px 4px;vertical-align:top;text-align:left;overflow-wrap:anywhere;word-break:break-word;"><div style="margin:0 0 4px 0;padding:3px 4px;border-left:3px solid #cbd5e1;background:#fff;border-radius:3px;"><div style="line-height:1.25;"><div style="font-weight:600;">GE1073</div><div>智慧財產權</div><div style="display:inline-block;margin-top:2px;padding:1px 4px;background:#f3f4f6;border-radius:3px;font-size:11px;">R1R2</div></div></div></td>
<td style="border-bottom:1px solid #eee;padding:6px 4px;vertical-align:top;text-align:left;overflow-wrap:anywhere;word-break:break-word;"></td>
<td style="border-bottom:1px solid #eee;padding:6px 4px;vertical-align:top;text-align:left;overflow-wrap:anywhere;word-break:break-word;"></td>
<td style="border-bottom:1px solid #eee;padding:6px 4px;vertical-align:top;text-align:left;overflow-wrap:anywhere;word-break:break-word;"></td>
</tr>
<tr style="background:#f7f7f7;">
<td style="border-bottom:1px solid #eee;padding:6px 4px;vertical-align:top;text-align:center;white-space:nowrap;font-weight:600;">2</td>
<td style="border-bottom:1px solid #eee;padding:6px 4px;vertical-align:top;text-align:center;white-space:nowrap;">09:00-09:50</td>
<td style="border-bottom:1px solid #eee;padding:6px 4px;vertical-align:top;text-align:left;overflow-wrap:anywhere;word-break:break-word;"></td>
<td style="border-bottom:1px solid #eee;padding:6px 4px;vertical-align:top;text-align:left;overflow-wrap:anywhere;word-break:break-word;"></td>
<td style="border-bottom:1px solid #eee;padding:6px 4px;vertical-align:top;text-align:left;overflow-wrap:anywhere;word-break:break-word;"><div style="margin:0 0 4px 0;padding:3px 4px;border-left:3px solid #cbd5e1;background:#fff;border-radius:3px;"><div style="line-height:1.25;"><div style="font-weight:600;">EE2140</div><div>電磁學</div><div style="display:inline-block;margin-top:2px;padding:1px 4px;background:#f3f4f6;border-radius:3px;font-size:11px;">M3M4W2</div></div></div></td>
<td style="border-bottom:1px solid #eee;padding:6px 4px;vertical-align:top;text-align:left;overflow-wrap:anywhere;word-break:break-word;"><div style="margin:0 0 4px 0;padding:3px 4px;border-left:3px solid #cbd5e1;background:#fff;border-radius:3px;"><div style="line-height:1.25;"><div style="font-weight:600;">GE1073</div><div>智慧財產權</div><div style="display:inline-block;margin-top:2px;padding:1px 4px;background:#f3f4f6;border-radius:3px;font-size:11px;">R1R2</div></div></div></td>
<td style="border-bottom:1px solid #eee;padding:6px 4px;vertical-align:top;text-align:left;overflow-wrap:anywhere;word-break:break-word;"></td>
<td style="border-bottom:1px solid #eee;padding:6px 4px;vertical-align:top;text-align:left;overflow-wrap:anywhere;word-break:break-word;"></td>
<td style="border-bottom:1px solid #eee;padding:6px 4px;vertical-align:top;text-align:left;overflow-wrap:anywhere;word-break:break-word;"></td>
</tr>
<tr style="background:#fff;">
<td style="border-bottom:1px solid #eee;padding:6px 4px;vertical-align:top;text-align:center;white-space:nowrap;font-weight:600;">3</td>
<td style="border-bottom:1px solid #eee;padding:6px 4px;vertical-align:top;text-align:center;white-space:nowrap;">10:10-11:00</td>
<td style="border-bottom:1px solid #eee;padding:6px 4px;vertical-align:top;text-align:left;overflow-wrap:anywhere;word-break:break-word;"><div style="margin:0 0 4px 0;padding:3px 4px;border-left:3px solid #cbd5e1;background:#fff;border-radius:3px;"><div style="line-height:1.25;"><div style="font-weight:600;">EE2140</div><div>電磁學</div><div style="display:inline-block;margin-top:2px;padding:1px 4px;background:#f3f4f6;border-radius:3px;font-size:11px;">M3M4W2</div></div></div></td>
<td style="border-bottom:1px solid #eee;padding:6px 4px;vertical-align:top;text-align:left;overflow-wrap:anywhere;word-break:break-word;"><div style="margin:0 0 4px 0;padding:3px 4px;border-left:3px solid #cbd5e1;background:#fff;border-radius:3px;"><div style="line-height:1.25;"><div style="font-weight:600;">EE2020</div><div>偏微分方程與複變函數</div><div style="display:inline-block;margin-top:2px;padding:1px 4px;background:#f3f4f6;border-radius:3px;font-size:11px;">T3T4R3R4</div></div></div></td>
<td style="border-bottom:1px solid #eee;padding:6px 4px;vertical-align:top;text-align:left;overflow-wrap:anywhere;word-break:break-word;"></td>
<td style="border-bottom:1px solid #eee;padding:6px 4px;vertical-align:top;text-align:left;overflow-wrap:anywhere;word-break:break-word;"><div style="margin:0 0 4px 0;padding:3px 4px;border-left:3px solid #cbd5e1;background:#fff;border-radius:3px;"><div style="line-height:1.25;"><div style="font-weight:600;">EE2020</div><div>偏微分方程與複變函數</div><div style="display:inline-block;margin-top:2px;padding:1px 4px;background:#f3f4f6;border-radius:3px;font-size:11px;">T3T4R3R4</div></div></div></td>
<td style="border-bottom:1px solid #eee;padding:6px 4px;vertical-align:top;text-align:left;overflow-wrap:anywhere;word-break:break-word;"></td>
<td style="border-bottom:1px solid #eee;padding:6px 4px;vertical-align:top;text-align:left;overflow-wrap:anywhere;word-break:break-word;"></td>
<td style="border-bottom:1px solid #eee;padding:6px 4px;vertical-align:top;text-align:left;overflow-wrap:anywhere;word-break:break-word;"></td>
</tr>
<tr style="background:#f7f7f7;">
<td style="border-bottom:1px solid #eee;padding:6px 4px;vertical-align:top;text-align:center;white-space:nowrap;font-weight:600;">4</td>
<td style="border-bottom:1px solid #eee;padding:6px 4px;vertical-align:top;text-align:center;white-space:nowrap;">11:10-12:00</td>
<td style="border-bottom:1px solid #eee;padding:6px 4px;vertical-align:top;text-align:left;overflow-wrap:anywhere;word-break:break-word;"><div style="margin:0 0 4px 0;padding:3px 4px;border-left:3px solid #cbd5e1;background:#fff;border-radius:3px;"><div style="line-height:1.25;"><div style="font-weight:600;">EE2140</div><div>電磁學</div><div style="display:inline-block;margin-top:2px;padding:1px 4px;background:#f3f4f6;border-radius:3px;font-size:11px;">M3M4W2</div></div></div></td>
<td style="border-bottom:1px solid #eee;padding:6px 4px;vertical-align:top;text-align:left;overflow-wrap:anywhere;word-break:break-word;"><div style="margin:0 0 4px 0;padding:3px 4px;border-left:3px solid #cbd5e1;background:#fff;border-radius:3px;"><div style="line-height:1.25;"><div style="font-weight:600;">EE2020</div><div>偏微分方程與複變函數</div><div style="display:inline-block;margin-top:2px;padding:1px 4px;background:#f3f4f6;border-radius:3px;font-size:11px;">T3T4R3R4</div></div></div></td>
<td style="border-bottom:1px solid #eee;padding:6px 4px;vertical-align:top;text-align:left;overflow-wrap:anywhere;word-break:break-word;"></td>
<td style="border-bottom:1px solid #eee;padding:6px 4px;vertical-align:top;text-align:left;overflow-wrap:anywhere;word-break:break-word;"><div style="margin:0 0 4px 0;padding:3px 4px;border-left:3px solid #cbd5e1;background:#fff;border-radius:3px;"><div style="line-height:1.25;"><div style="font-weight:600;">EE2020</div><div>偏微分方程與複變函數</div><div style="display:inline-block;margin-top:2px;padding:1px 4px;background:#f3f4f6;border-radius:3px;font-size:11px;">T3T4R3R4</div></div></div></td>
<td style="border-bottom:1px solid #eee;padding:6px 4px;vertical-align:top;text-align:left;overflow-wrap:anywhere;word-break:break-word;"></td>
<td style="border-bottom:1px solid #eee;padding:6px 4px;vertical-align:top;text-align:left;overflow-wrap:anywhere;word-break:break-word;"></td>
<td style="border-bottom:1px solid #eee;padding:6px 4px;vertical-align:top;text-align:left;overflow-wrap:anywhere;word-break:break-word;"></td>
</tr>
<tr style="background:#fff;">
<td style="border-bottom:1px solid #eee;padding:6px 4px;vertical-align:top;text-align:center;white-space:nowrap;font-weight:600;">n</td>
<td style="border-bottom:1px solid #eee;padding:6px 4px;vertical-align:top;text-align:center;white-space:nowrap;">12:10-13:00</td>
<td style="border-bottom:1px solid #eee;padding:6px 4px;vertical-align:top;text-align:left;overflow-wrap:anywhere;word-break:break-word;"></td>
<td style="border-bottom:1px solid #eee;padding:6px 4px;vertical-align:top;text-align:left;overflow-wrap:anywhere;word-break:break-word;"></td>
<td style="border-bottom:1px solid #eee;padding:6px 4px;vertical-align:top;text-align:left;overflow-wrap:anywhere;word-break:break-word;"><div style="margin:0 0 4px 0;padding:3px 4px;border-left:3px solid #cbd5e1;background:#fff;border-radius:3px;"><div style="line-height:1.25;"><div style="font-weight:600;">GE1026</div><div>紀錄片賞析與創作</div><div style="display:inline-block;margin-top:2px;padding:1px 4px;background:#f3f4f6;border-radius:3px;font-size:11px;">WnW5W6</div></div></div></td>
<td style="border-bottom:1px solid #eee;padding:6px 4px;vertical-align:top;text-align:left;overflow-wrap:anywhere;word-break:break-word;"></td>
<td style="border-bottom:1px solid #eee;padding:6px 4px;vertical-align:top;text-align:left;overflow-wrap:anywhere;word-break:break-word;"></td>
<td style="border-bottom:1px solid #eee;padding:6px 4px;vertical-align:top;text-align:left;overflow-wrap:anywhere;word-break:break-word;"></td>
<td style="border-bottom:1px solid #eee;padding:6px 4px;vertical-align:top;text-align:left;overflow-wrap:anywhere;word-break:break-word;"></td>
</tr>
<tr style="background:#f7f7f7;">
<td style="border-bottom:1px solid #eee;padding:6px 4px;vertical-align:top;text-align:center;white-space:nowrap;font-weight:600;">5</td>
<td style="border-bottom:1px solid #eee;padding:6px 4px;vertical-align:top;text-align:center;white-space:nowrap;">13:20-14:10</td>
<td style="border-bottom:1px solid #eee;padding:6px 4px;vertical-align:top;text-align:left;overflow-wrap:anywhere;word-break:break-word;"><div style="margin:0 0 4px 0;padding:3px 4px;border-left:3px solid #cbd5e1;background:#fff;border-radius:3px;"><div style="line-height:1.25;"><div style="font-weight:600;">PE1120</div><div>重量訓練</div><div style="display:inline-block;margin-top:2px;padding:1px 4px;background:#f3f4f6;border-radius:3px;font-size:11px;">M5M6</div></div></div></td>
<td style="border-bottom:1px solid #eee;padding:6px 4px;vertical-align:top;text-align:left;overflow-wrap:anywhere;word-break:break-word;"><div style="margin:0 0 4px 0;padding:3px 4px;border-left:3px solid #cbd5e1;background:#fff;border-radius:3px;"><div style="line-height:1.25;"><div style="font-weight:600;">EE3060</div><div>機率</div><div style="display:inline-block;margin-top:2px;padding:1px 4px;background:#f3f4f6;border-radius:3px;font-size:11px;">T5T6R5R6</div></div></div></td>
<td style="border-bottom:1px solid #eee;padding:6px 4px;vertical-align:top;text-align:left;overflow-wrap:anywhere;word-break:break-word;"><div style="margin:0 0 4px 0;padding:3px 4px;border-left:3px solid #cbd5e1;background:#fff;border-radius:3px;"><div style="line-height:1.25;"><div style="font-weight:600;">GE1026</div><div>紀錄片賞析與創作</div><div style="display:inline-block;margin-top:2px;padding:1px 4px;background:#f3f4f6;border-radius:3px;font-size:11px;">WnW5W6</div></div></div></td>
<td style="border-bottom:1px solid #eee;padding:6px 4px;vertical-align:top;text-align:left;overflow-wrap:anywhere;word-break:break-word;"><div style="margin:0 0 4px 0;padding:3px 4px;border-left:3px solid #cbd5e1;background:#fff;border-radius:3px;"><div style="line-height:1.25;"><div style="font-weight:600;">EE3060</div><div>機率</div><div style="display:inline-block;margin-top:2px;padding:1px 4px;background:#f3f4f6;border-radius:3px;font-size:11px;">T5T6R5R6</div></div></div></td>
<td style="border-bottom:1px solid #eee;padding:6px 4px;vertical-align:top;text-align:left;overflow-wrap:anywhere;word-break:break-word;"></td>
<td style="border-bottom:1px solid #eee;padding:6px 4px;vertical-align:top;text-align:left;overflow-wrap:anywhere;word-break:break-word;"></td>
<td style="border-bottom:1px solid #eee;padding:6px 4px;vertical-align:top;text-align:left;overflow-wrap:anywhere;word-break:break-word;"></td>
</tr>
<tr style="background:#fff;">
<td style="border-bottom:1px solid #eee;padding:6px 4px;vertical-align:top;text-align:center;white-space:nowrap;font-weight:600;">6</td>
<td style="border-bottom:1px solid #eee;padding:6px 4px;vertical-align:top;text-align:center;white-space:nowrap;">14:20-15:10</td>
<td style="border-bottom:1px solid #eee;padding:6px 4px;vertical-align:top;text-align:left;overflow-wrap:anywhere;word-break:break-word;"><div style="margin:0 0 4px 0;padding:3px 4px;border-left:3px solid #cbd5e1;background:#fff;border-radius:3px;"><div style="line-height:1.25;"><div style="font-weight:600;">PE1120</div><div>重量訓練</div><div style="display:inline-block;margin-top:2px;padding:1px 4px;background:#f3f4f6;border-radius:3px;font-size:11px;">M5M6</div></div></div></td>
<td style="border-bottom:1px solid #eee;padding:6px 4px;vertical-align:top;text-align:left;overflow-wrap:anywhere;word-break:break-word;"><div style="margin:0 0 4px 0;padding:3px 4px;border-left:3px solid #cbd5e1;background:#fff;border-radius:3px;"><div style="line-height:1.25;"><div style="font-weight:600;">EE3060</div><div>機率</div><div style="display:inline-block;margin-top:2px;padding:1px 4px;background:#f3f4f6;border-radius:3px;font-size:11px;">T5T6R5R6</div></div></div></td>
<td style="border-bottom:1px solid #eee;padding:6px 4px;vertical-align:top;text-align:left;overflow-wrap:anywhere;word-break:break-word;"><div style="margin:0 0 4px 0;padding:3px 4px;border-left:3px solid #cbd5e1;background:#fff;border-radius:3px;"><div style="line-height:1.25;"><div style="font-weight:600;">GE1026</div><div>紀錄片賞析與創作</div><div style="display:inline-block;margin-top:2px;padding:1px 4px;background:#f3f4f6;border-radius:3px;font-size:11px;">WnW5W6</div></div></div></td>
<td style="border-bottom:1px solid #eee;padding:6px 4px;vertical-align:top;text-align:left;overflow-wrap:anywhere;word-break:break-word;"><div style="margin:0 0 4px 0;padding:3px 4px;border-left:3px solid #cbd5e1;background:#fff;border-radius:3px;"><div style="line-height:1.25;"><div style="font-weight:600;">EE3060</div><div>機率</div><div style="display:inline-block;margin-top:2px;padding:1px 4px;background:#f3f4f6;border-radius:3px;font-size:11px;">T5T6R5R6</div></div></div></td>
<td style="border-bottom:1px solid #eee;padding:6px 4px;vertical-align:top;text-align:left;overflow-wrap:anywhere;word-break:break-word;"></td>
<td style="border-bottom:1px solid #eee;padding:6px 4px;vertical-align:top;text-align:left;overflow-wrap:anywhere;word-break:break-word;"></td>
<td style="border-bottom:1px solid #eee;padding:6px 4px;vertical-align:top;text-align:left;overflow-wrap:anywhere;word-break:break-word;"></td>
</tr>
<tr style="background:#f7f7f7;">
<td style="border-bottom:1px solid #eee;padding:6px 4px;vertical-align:top;text-align:center;white-space:nowrap;font-weight:600;">7</td>
<td style="border-bottom:1px solid #eee;padding:6px 4px;vertical-align:top;text-align:center;white-space:nowrap;">15:30-16:20</td>
<td style="border-bottom:1px solid #eee;padding:6px 4px;vertical-align:top;text-align:left;overflow-wrap:anywhere;word-break:break-word;"></td>
<td style="border-bottom:1px solid #eee;padding:6px 4px;vertical-align:top;text-align:left;overflow-wrap:anywhere;word-break:break-word;"></td>
<td style="border-bottom:1px solid #eee;padding:6px 4px;vertical-align:top;text-align:left;overflow-wrap:anywhere;word-break:break-word;"><div style="margin:0 0 4px 0;padding:3px 4px;border-left:3px solid #cbd5e1;background:#fff;border-radius:3px;"><div style="line-height:1.25;"><div style="font-weight:600;">EE2405</div><div>嵌入式系統與實驗</div><div style="display:inline-block;margin-top:2px;padding:1px 4px;background:#f3f4f6;border-radius:3px;font-size:11px;">W7W8W9WaWb</div></div></div></td>
<td style="border-bottom:1px solid #eee;padding:6px 4px;vertical-align:top;text-align:left;overflow-wrap:anywhere;word-break:break-word;"></td>
<td style="border-bottom:1px solid #eee;padding:6px 4px;vertical-align:top;text-align:left;overflow-wrap:anywhere;word-break:break-word;"></td>
<td style="border-bottom:1px solid #eee;padding:6px 4px;vertical-align:top;text-align:left;overflow-wrap:anywhere;word-break:break-word;"></td>
<td style="border-bottom:1px solid #eee;padding:6px 4px;vertical-align:top;text-align:left;overflow-wrap:anywhere;word-break:break-word;"></td>
</tr>
<tr style="background:#fff;">
<td style="border-bottom:1px solid #eee;padding:6px 4px;vertical-align:top;text-align:center;white-space:nowrap;font-weight:600;">8</td>
<td style="border-bottom:1px solid #eee;padding:6px 4px;vertical-align:top;text-align:center;white-space:nowrap;">16:30-17:20</td>
<td style="border-bottom:1px solid #eee;padding:6px 4px;vertical-align:top;text-align:left;overflow-wrap:anywhere;word-break:break-word;"></td>
<td style="border-bottom:1px solid #eee;padding:6px 4px;vertical-align:top;text-align:left;overflow-wrap:anywhere;word-break:break-word;"></td>
<td style="border-bottom:1px solid #eee;padding:6px 4px;vertical-align:top;text-align:left;overflow-wrap:anywhere;word-break:break-word;"><div style="margin:0 0 4px 0;padding:3px 4px;border-left:3px solid #cbd5e1;background:#fff;border-radius:3px;"><div style="line-height:1.25;"><div style="font-weight:600;">EE2405</div><div>嵌入式系統與實驗</div><div style="display:inline-block;margin-top:2px;padding:1px 4px;background:#f3f4f6;border-radius:3px;font-size:11px;">W7W8W9WaWb</div></div></div></td>
<td style="border-bottom:1px solid #eee;padding:6px 4px;vertical-align:top;text-align:left;overflow-wrap:anywhere;word-break:break-word;"></td>
<td style="border-bottom:1px solid #eee;padding:6px 4px;vertical-align:top;text-align:left;overflow-wrap:anywhere;word-break:break-word;"></td>
<td style="border-bottom:1px solid #eee;padding:6px 4px;vertical-align:top;text-align:left;overflow-wrap:anywhere;word-break:break-word;"><div style="margin:0 0 4px 0;padding:3px 4px;border-left:3px solid #cbd5e1;background:#fff;border-radius:3px;"><div style="line-height:1.25;"><div style="font-weight:600;">EE3900</div><div>實作專題一</div><div style="display:inline-block;margin-top:2px;padding:1px 4px;background:#f3f4f6;border-radius:3px;font-size:11px;">S8</div></div></div></td>
<td style="border-bottom:1px solid #eee;padding:6px 4px;vertical-align:top;text-align:left;overflow-wrap:anywhere;word-break:break-word;"></td>
</tr>
<tr style="background:#f7f7f7;">
<td style="border-bottom:1px solid #eee;padding:6px 4px;vertical-align:top;text-align:center;white-space:nowrap;font-weight:600;">9</td>
<td style="border-bottom:1px solid #eee;padding:6px 4px;vertical-align:top;text-align:center;white-space:nowrap;">17:30-18:20</td>
<td style="border-bottom:1px solid #eee;padding:6px 4px;vertical-align:top;text-align:left;overflow-wrap:anywhere;word-break:break-word;"></td>
<td style="border-bottom:1px solid #eee;padding:6px 4px;vertical-align:top;text-align:left;overflow-wrap:anywhere;word-break:break-word;"></td>
<td style="border-bottom:1px solid #eee;padding:6px 4px;vertical-align:top;text-align:left;overflow-wrap:anywhere;word-break:break-word;"><div style="margin:0 0 4px 0;padding:3px 4px;border-left:3px solid #cbd5e1;background:#fff;border-radius:3px;"><div style="line-height:1.25;"><div style="font-weight:600;">EE2405</div><div>嵌入式系統與實驗</div><div style="display:inline-block;margin-top:2px;padding:1px 4px;background:#f3f4f6;border-radius:3px;font-size:11px;">W7W8W9WaWb</div></div></div></td>
<td style="border-bottom:1px solid #eee;padding:6px 4px;vertical-align:top;text-align:left;overflow-wrap:anywhere;word-break:break-word;"></td>
<td style="border-bottom:1px solid #eee;padding:6px 4px;vertical-align:top;text-align:left;overflow-wrap:anywhere;word-break:break-word;"></td>
<td style="border-bottom:1px solid #eee;padding:6px 4px;vertical-align:top;text-align:left;overflow-wrap:anywhere;word-break:break-word;"></td>
<td style="border-bottom:1px solid #eee;padding:6px 4px;vertical-align:top;text-align:left;overflow-wrap:anywhere;word-break:break-word;"></td>
</tr>
<tr style="background:#fff;">
<td style="border-bottom:1px solid #eee;padding:6px 4px;vertical-align:top;text-align:center;white-space:nowrap;font-weight:600;">a</td>
<td style="border-bottom:1px solid #eee;padding:6px 4px;vertical-align:top;text-align:center;white-space:nowrap;">18:30-19:20</td>
<td style="border-bottom:1px solid #eee;padding:6px 4px;vertical-align:top;text-align:left;overflow-wrap:anywhere;word-break:break-word;"></td>
<td style="border-bottom:1px solid #eee;padding:6px 4px;vertical-align:top;text-align:left;overflow-wrap:anywhere;word-break:break-word;"><div style="margin:0 0 4px 0;padding:3px 4px;border-left:3px solid #cbd5e1;background:#fff;border-radius:3px;"><div style="line-height:1.25;"><div style="font-weight:600;">GE1024</div><div>產業創新實作</div><div style="display:inline-block;margin-top:2px;padding:1px 4px;background:#f3f4f6;border-radius:3px;font-size:11px;">TaTb</div></div></div></td>
<td style="border-bottom:1px solid #eee;padding:6px 4px;vertical-align:top;text-align:left;overflow-wrap:anywhere;word-break:break-word;"><div style="margin:0 0 4px 0;padding:3px 4px;border-left:3px solid #cbd5e1;background:#fff;border-radius:3px;"><div style="line-height:1.25;"><div style="font-weight:600;">EE2405</div><div>嵌入式系統與實驗</div><div style="display:inline-block;margin-top:2px;padding:1px 4px;background:#f3f4f6;border-radius:3px;font-size:11px;">W7W8W9WaWb</div></div></div></td>
<td style="border-bottom:1px solid #eee;padding:6px 4px;vertical-align:top;text-align:left;overflow-wrap:anywhere;word-break:break-word;"></td>
<td style="border-bottom:1px solid #eee;padding:6px 4px;vertical-align:top;text-align:left;overflow-wrap:anywhere;word-break:break-word;"></td>
<td style="border-bottom:1px solid #eee;padding:6px 4px;vertical-align:top;text-align:left;overflow-wrap:anywhere;word-break:break-word;"></td>
<td style="border-bottom:1px solid #eee;padding:6px 4px;vertical-align:top;text-align:left;overflow-wrap:anywhere;word-break:break-word;"></td>
</tr>
<tr style="background:#f7f7f7;">
<td style="border-bottom:1px solid #eee;padding:6px 4px;vertical-align:top;text-align:center;white-space:nowrap;font-weight:600;">b</td>
<td style="border-bottom:1px solid #eee;padding:6px 4px;vertical-align:top;text-align:center;white-space:nowrap;">19:30-20:20</td>
<td style="border-bottom:1px solid #eee;padding:6px 4px;vertical-align:top;text-align:left;overflow-wrap:anywhere;word-break:break-word;"></td>
<td style="border-bottom:1px solid #eee;padding:6px 4px;vertical-align:top;text-align:left;overflow-wrap:anywhere;word-break:break-word;"><div style="margin:0 0 4px 0;padding:3px 4px;border-left:3px solid #cbd5e1;background:#fff;border-radius:3px;"><div style="line-height:1.25;"><div style="font-weight:600;">GE1024</div><div>產業創新實作</div><div style="display:inline-block;margin-top:2px;padding:1px 4px;background:#f3f4f6;border-radius:3px;font-size:11px;">TaTb</div></div></div></td>
<td style="border-bottom:1px solid #eee;padding:6px 4px;vertical-align:top;text-align:left;overflow-wrap:anywhere;word-break:break-word;"><div style="margin:0 0 4px 0;padding:3px 4px;border-left:3px solid #cbd5e1;background:#fff;border-radius:3px;"><div style="line-height:1.25;"><div style="font-weight:600;">EE2405</div><div>嵌入式系統與實驗</div><div style="display:inline-block;margin-top:2px;padding:1px 4px;background:#f3f4f6;border-radius:3px;font-size:11px;">W7W8W9WaWb</div></div></div></td>
<td style="border-bottom:1px solid #eee;padding:6px 4px;vertical-align:top;text-align:left;overflow-wrap:anywhere;word-break:break-word;"></td>
<td style="border-bottom:1px solid #eee;padding:6px 4px;vertical-align:top;text-align:left;overflow-wrap:anywhere;word-break:break-word;"></td>
<td style="border-bottom:1px solid #eee;padding:6px 4px;vertical-align:top;text-align:left;overflow-wrap:anywhere;word-break:break-word;"></td>
<td style="border-bottom:1px solid #eee;padding:6px 4px;vertical-align:top;text-align:left;overflow-wrap:anywhere;word-break:break-word;"></td>
</tr>
<tr style="background:#fff;">
<td style="border-bottom:1px solid #eee;padding:6px 4px;vertical-align:top;text-align:center;white-space:nowrap;font-weight:600;">c</td>
<td style="border-bottom:1px solid #eee;padding:6px 4px;vertical-align:top;text-align:center;white-space:nowrap;">20:30-21:20</td>
<td style="border-bottom:1px solid #eee;padding:6px 4px;vertical-align:top;text-align:left;overflow-wrap:anywhere;word-break:break-word;"></td>
<td style="border-bottom:1px solid #eee;padding:6px 4px;vertical-align:top;text-align:left;overflow-wrap:anywhere;word-break:break-word;"></td>
<td style="border-bottom:1px solid #eee;padding:6px 4px;vertical-align:top;text-align:left;overflow-wrap:anywhere;word-break:break-word;"></td>
<td style="border-bottom:1px solid #eee;padding:6px 4px;vertical-align:top;text-align:left;overflow-wrap:anywhere;word-break:break-word;"></td>
<td style="border-bottom:1px solid #eee;padding:6px 4px;vertical-align:top;text-align:left;overflow-wrap:anywhere;word-break:break-word;"></td>
<td style="border-bottom:1px solid #eee;padding:6px 4px;vertical-align:top;text-align:left;overflow-wrap:anywhere;word-break:break-word;"></td>
<td style="border-bottom:1px solid #eee;padding:6px 4px;vertical-align:top;text-align:left;overflow-wrap:anywhere;word-break:break-word;"></td>
</tr>
<tr style="background:#f7f7f7;">
<td style="border-bottom:1px solid #eee;padding:6px 4px;vertical-align:top;text-align:center;white-space:nowrap;font-weight:600;">d</td>
<td style="border-bottom:1px solid #eee;padding:6px 4px;vertical-align:top;text-align:center;white-space:nowrap;">21:30-22:20</td>
<td style="border-bottom:1px solid #eee;padding:6px 4px;vertical-align:top;text-align:left;overflow-wrap:anywhere;word-break:break-word;"></td>
<td style="border-bottom:1px solid #eee;padding:6px 4px;vertical-align:top;text-align:left;overflow-wrap:anywhere;word-break:break-word;"></td>
<td style="border-bottom:1px solid #eee;padding:6px 4px;vertical-align:top;text-align:left;overflow-wrap:anywhere;word-break:break-word;"></td>
<td style="border-bottom:1px solid #eee;padding:6px 4px;vertical-align:top;text-align:left;overflow-wrap:anywhere;word-break:break-word;"></td>
<td style="border-bottom:1px solid #eee;padding:6px 4px;vertical-align:top;text-align:left;overflow-wrap:anywhere;word-break:break-word;"></td>
<td style="border-bottom:1px solid #eee;padding:6px 4px;vertical-align:top;text-align:left;overflow-wrap:anywhere;word-break:break-word;"></td>
<td style="border-bottom:1px solid #eee;padding:6px 4px;vertical-align:top;text-align:left;overflow-wrap:anywhere;word-break:break-word;"></td>
</tr>
</tbody>
</table>

#### 學分狀態
- 目前總學分：**20**
- 正常學分範圍：**16-25**
- 低修：**False**
- 超修：**False**

#### 重要提醒
- 修課中的課只是在規劃模式假設會通過；如果沒有通過，畢業進度和推薦要重算。
- 這不是正式畢業審查，最後仍要以系辦確認為準。

#### 時間代碼說明
##### 星期代碼
| 代碼 | 星期 | English |
|---|---|---|
| `M` | 星期一 | Monday |
| `T` | 星期二 | Tuesday |
| `W` | 星期三 | Wednesday |
| `R` | 星期四 | Thursday |
| `F` | 星期五 | Friday |
| `S` | 星期六 | Saturday |
| `U` | 星期日 | Sunday |

##### 節次時間
| 節次 | 時間 |
|---|---|
| `1` | 08:00-08:50 |
| `2` | 09:00-09:50 |
| `3` | 10:10-11:00 |
| `4` | 11:10-12:00 |
| `n` | 12:10-13:00 |
| `5` | 13:20-14:10 |
| `6` | 14:20-15:10 |
| `7` | 15:30-16:20 |
| `8` | 16:30-17:20 |
| `9` | 17:30-18:20 |
| `a` | 18:30-19:20 |
| `b` | 19:30-20:20 |
| `c` | 20:30-21:20 |
| `d` | 21:30-22:20 |

例：`T3T4R3R4` 代表星期二第 3、4 節，和星期四第 3、4 節。


請輸入你的需求或修改意見： done


### 已確認課表

### 114 第二學期課表建議

#### 推薦課程與理由
| 課號 | 課名 | 授課老師 | 學分 | 時間 | 為什麼建議修 | 評價參考 |
|---|---|---|---:|---|---|---|
| EE2020 | 偏微分方程與複變函數 | 陳國平 | 3 | `T3T4R3R4` | 補 EE112 必修缺口。 |  |
| EE2140 | 電磁學 | 黃承彬 | 3 | `M3M4W2` | 補 EE112 必修缺口。 |  |
| EE3900 | 實作專題一 | 指導教授 | 1 | `S8` | 補 EE112 必修缺口。 |  |
| EE3060 | 機率 | 謝欣霖 | 3 | `T5T6R5R6` | 補機率替代必修需求。 |  |
| EE2405 | 嵌入式系統與實驗 | 劉靖家 | 3 | `W7W8W9WaWb` | 依照初始 EE-first 排課策略，在未排除實驗課時加入 1 門電機實驗課。 |  |
| GE1024 | 產業創新實作 | 商雅婷,李佳玫,蔡協亨 | 2 | `TaTb` | 用 GE/GEC 通識課補其餘選修與學分。 |  |
| GE1026 | 紀錄片賞析與創作 | 蕭菊貞 | 3 | `WnW5W6` | 用 GE/GEC 通識課補其餘選修與學分。 |  |
| GE1073 | 智慧財產權 | 楊智傑 | 2 | `R1R2` | 用 GE/GEC 通識課補其餘選修與學分。 |  |
| PE1120 | 重量訓練 | 李柏均 | 0 | `M5M6` | 體育課可放進課表平衡生活節奏，但這門是 0 學分，不計入畢業總學分。 |  |

#### 每週課表
<table style="width:100%;table-layout:fixed;border-collapse:collapse;font-size:13px;line-height:1.25;">
<colgroup>
<col style="width:6%;">
<col style="width:13%;">
<col style="width:11.57%;">
<col style="width:11.57%;">
<col style="width:11.57%;">
<col style="width:11.57%;">
<col style="width:11.57%;">
<col style="width:11.57%;">
<col style="width:11.57%;">
</colgroup>
<thead>
<tr>
<th style="border-bottom:1px solid #bbb;padding:6px 4px;text-align:center;vertical-align:middle;background:#fff;font-weight:700;">節次</th>
<th style="border-bottom:1px solid #bbb;padding:6px 4px;text-align:center;vertical-align:middle;background:#fff;font-weight:700;">時間</th>
<th style="border-bottom:1px solid #bbb;padding:6px 4px;text-align:center;vertical-align:middle;background:#fff;font-weight:700;">M<br>一</th>
<th style="border-bottom:1px solid #bbb;padding:6px 4px;text-align:center;vertical-align:middle;background:#fff;font-weight:700;">T<br>二</th>
<th style="border-bottom:1px solid #bbb;padding:6px 4px;text-align:center;vertical-align:middle;background:#fff;font-weight:700;">W<br>三</th>
<th style="border-bottom:1px solid #bbb;padding:6px 4px;text-align:center;vertical-align:middle;background:#fff;font-weight:700;">R<br>四</th>
<th style="border-bottom:1px solid #bbb;padding:6px 4px;text-align:center;vertical-align:middle;background:#fff;font-weight:700;">F<br>五</th>
<th style="border-bottom:1px solid #bbb;padding:6px 4px;text-align:center;vertical-align:middle;background:#fff;font-weight:700;">S<br>六</th>
<th style="border-bottom:1px solid #bbb;padding:6px 4px;text-align:center;vertical-align:middle;background:#fff;font-weight:700;">U<br>日</th>
</tr>
</thead>
<tbody>
<tr style="background:#fff;">
<td style="border-bottom:1px solid #eee;padding:6px 4px;vertical-align:top;text-align:center;white-space:nowrap;font-weight:600;">1</td>
<td style="border-bottom:1px solid #eee;padding:6px 4px;vertical-align:top;text-align:center;white-space:nowrap;">08:00-08:50</td>
<td style="border-bottom:1px solid #eee;padding:6px 4px;vertical-align:top;text-align:left;overflow-wrap:anywhere;word-break:break-word;"></td>
<td style="border-bottom:1px solid #eee;padding:6px 4px;vertical-align:top;text-align:left;overflow-wrap:anywhere;word-break:break-word;"></td>
<td style="border-bottom:1px solid #eee;padding:6px 4px;vertical-align:top;text-align:left;overflow-wrap:anywhere;word-break:break-word;"></td>
<td style="border-bottom:1px solid #eee;padding:6px 4px;vertical-align:top;text-align:left;overflow-wrap:anywhere;word-break:break-word;"><div style="margin:0 0 4px 0;padding:3px 4px;border-left:3px solid #cbd5e1;background:#fff;border-radius:3px;"><div style="line-height:1.25;"><div style="font-weight:600;">GE1073</div><div>智慧財產權</div><div style="display:inline-block;margin-top:2px;padding:1px 4px;background:#f3f4f6;border-radius:3px;font-size:11px;">R1R2</div></div></div></td>
<td style="border-bottom:1px solid #eee;padding:6px 4px;vertical-align:top;text-align:left;overflow-wrap:anywhere;word-break:break-word;"></td>
<td style="border-bottom:1px solid #eee;padding:6px 4px;vertical-align:top;text-align:left;overflow-wrap:anywhere;word-break:break-word;"></td>
<td style="border-bottom:1px solid #eee;padding:6px 4px;vertical-align:top;text-align:left;overflow-wrap:anywhere;word-break:break-word;"></td>
</tr>
<tr style="background:#f7f7f7;">
<td style="border-bottom:1px solid #eee;padding:6px 4px;vertical-align:top;text-align:center;white-space:nowrap;font-weight:600;">2</td>
<td style="border-bottom:1px solid #eee;padding:6px 4px;vertical-align:top;text-align:center;white-space:nowrap;">09:00-09:50</td>
<td style="border-bottom:1px solid #eee;padding:6px 4px;vertical-align:top;text-align:left;overflow-wrap:anywhere;word-break:break-word;"></td>
<td style="border-bottom:1px solid #eee;padding:6px 4px;vertical-align:top;text-align:left;overflow-wrap:anywhere;word-break:break-word;"></td>
<td style="border-bottom:1px solid #eee;padding:6px 4px;vertical-align:top;text-align:left;overflow-wrap:anywhere;word-break:break-word;"><div style="margin:0 0 4px 0;padding:3px 4px;border-left:3px solid #cbd5e1;background:#fff;border-radius:3px;"><div style="line-height:1.25;"><div style="font-weight:600;">EE2140</div><div>電磁學</div><div style="display:inline-block;margin-top:2px;padding:1px 4px;background:#f3f4f6;border-radius:3px;font-size:11px;">M3M4W2</div></div></div></td>
<td style="border-bottom:1px solid #eee;padding:6px 4px;vertical-align:top;text-align:left;overflow-wrap:anywhere;word-break:break-word;"><div style="margin:0 0 4px 0;padding:3px 4px;border-left:3px solid #cbd5e1;background:#fff;border-radius:3px;"><div style="line-height:1.25;"><div style="font-weight:600;">GE1073</div><div>智慧財產權</div><div style="display:inline-block;margin-top:2px;padding:1px 4px;background:#f3f4f6;border-radius:3px;font-size:11px;">R1R2</div></div></div></td>
<td style="border-bottom:1px solid #eee;padding:6px 4px;vertical-align:top;text-align:left;overflow-wrap:anywhere;word-break:break-word;"></td>
<td style="border-bottom:1px solid #eee;padding:6px 4px;vertical-align:top;text-align:left;overflow-wrap:anywhere;word-break:break-word;"></td>
<td style="border-bottom:1px solid #eee;padding:6px 4px;vertical-align:top;text-align:left;overflow-wrap:anywhere;word-break:break-word;"></td>
</tr>
<tr style="background:#fff;">
<td style="border-bottom:1px solid #eee;padding:6px 4px;vertical-align:top;text-align:center;white-space:nowrap;font-weight:600;">3</td>
<td style="border-bottom:1px solid #eee;padding:6px 4px;vertical-align:top;text-align:center;white-space:nowrap;">10:10-11:00</td>
<td style="border-bottom:1px solid #eee;padding:6px 4px;vertical-align:top;text-align:left;overflow-wrap:anywhere;word-break:break-word;"><div style="margin:0 0 4px 0;padding:3px 4px;border-left:3px solid #cbd5e1;background:#fff;border-radius:3px;"><div style="line-height:1.25;"><div style="font-weight:600;">EE2140</div><div>電磁學</div><div style="display:inline-block;margin-top:2px;padding:1px 4px;background:#f3f4f6;border-radius:3px;font-size:11px;">M3M4W2</div></div></div></td>
<td style="border-bottom:1px solid #eee;padding:6px 4px;vertical-align:top;text-align:left;overflow-wrap:anywhere;word-break:break-word;"><div style="margin:0 0 4px 0;padding:3px 4px;border-left:3px solid #cbd5e1;background:#fff;border-radius:3px;"><div style="line-height:1.25;"><div style="font-weight:600;">EE2020</div><div>偏微分方程與複變函數</div><div style="display:inline-block;margin-top:2px;padding:1px 4px;background:#f3f4f6;border-radius:3px;font-size:11px;">T3T4R3R4</div></div></div></td>
<td style="border-bottom:1px solid #eee;padding:6px 4px;vertical-align:top;text-align:left;overflow-wrap:anywhere;word-break:break-word;"></td>
<td style="border-bottom:1px solid #eee;padding:6px 4px;vertical-align:top;text-align:left;overflow-wrap:anywhere;word-break:break-word;"><div style="margin:0 0 4px 0;padding:3px 4px;border-left:3px solid #cbd5e1;background:#fff;border-radius:3px;"><div style="line-height:1.25;"><div style="font-weight:600;">EE2020</div><div>偏微分方程與複變函數</div><div style="display:inline-block;margin-top:2px;padding:1px 4px;background:#f3f4f6;border-radius:3px;font-size:11px;">T3T4R3R4</div></div></div></td>
<td style="border-bottom:1px solid #eee;padding:6px 4px;vertical-align:top;text-align:left;overflow-wrap:anywhere;word-break:break-word;"></td>
<td style="border-bottom:1px solid #eee;padding:6px 4px;vertical-align:top;text-align:left;overflow-wrap:anywhere;word-break:break-word;"></td>
<td style="border-bottom:1px solid #eee;padding:6px 4px;vertical-align:top;text-align:left;overflow-wrap:anywhere;word-break:break-word;"></td>
</tr>
<tr style="background:#f7f7f7;">
<td style="border-bottom:1px solid #eee;padding:6px 4px;vertical-align:top;text-align:center;white-space:nowrap;font-weight:600;">4</td>
<td style="border-bottom:1px solid #eee;padding:6px 4px;vertical-align:top;text-align:center;white-space:nowrap;">11:10-12:00</td>
<td style="border-bottom:1px solid #eee;padding:6px 4px;vertical-align:top;text-align:left;overflow-wrap:anywhere;word-break:break-word;"><div style="margin:0 0 4px 0;padding:3px 4px;border-left:3px solid #cbd5e1;background:#fff;border-radius:3px;"><div style="line-height:1.25;"><div style="font-weight:600;">EE2140</div><div>電磁學</div><div style="display:inline-block;margin-top:2px;padding:1px 4px;background:#f3f4f6;border-radius:3px;font-size:11px;">M3M4W2</div></div></div></td>
<td style="border-bottom:1px solid #eee;padding:6px 4px;vertical-align:top;text-align:left;overflow-wrap:anywhere;word-break:break-word;"><div style="margin:0 0 4px 0;padding:3px 4px;border-left:3px solid #cbd5e1;background:#fff;border-radius:3px;"><div style="line-height:1.25;"><div style="font-weight:600;">EE2020</div><div>偏微分方程與複變函數</div><div style="display:inline-block;margin-top:2px;padding:1px 4px;background:#f3f4f6;border-radius:3px;font-size:11px;">T3T4R3R4</div></div></div></td>
<td style="border-bottom:1px solid #eee;padding:6px 4px;vertical-align:top;text-align:left;overflow-wrap:anywhere;word-break:break-word;"></td>
<td style="border-bottom:1px solid #eee;padding:6px 4px;vertical-align:top;text-align:left;overflow-wrap:anywhere;word-break:break-word;"><div style="margin:0 0 4px 0;padding:3px 4px;border-left:3px solid #cbd5e1;background:#fff;border-radius:3px;"><div style="line-height:1.25;"><div style="font-weight:600;">EE2020</div><div>偏微分方程與複變函數</div><div style="display:inline-block;margin-top:2px;padding:1px 4px;background:#f3f4f6;border-radius:3px;font-size:11px;">T3T4R3R4</div></div></div></td>
<td style="border-bottom:1px solid #eee;padding:6px 4px;vertical-align:top;text-align:left;overflow-wrap:anywhere;word-break:break-word;"></td>
<td style="border-bottom:1px solid #eee;padding:6px 4px;vertical-align:top;text-align:left;overflow-wrap:anywhere;word-break:break-word;"></td>
<td style="border-bottom:1px solid #eee;padding:6px 4px;vertical-align:top;text-align:left;overflow-wrap:anywhere;word-break:break-word;"></td>
</tr>
<tr style="background:#fff;">
<td style="border-bottom:1px solid #eee;padding:6px 4px;vertical-align:top;text-align:center;white-space:nowrap;font-weight:600;">n</td>
<td style="border-bottom:1px solid #eee;padding:6px 4px;vertical-align:top;text-align:center;white-space:nowrap;">12:10-13:00</td>
<td style="border-bottom:1px solid #eee;padding:6px 4px;vertical-align:top;text-align:left;overflow-wrap:anywhere;word-break:break-word;"></td>
<td style="border-bottom:1px solid #eee;padding:6px 4px;vertical-align:top;text-align:left;overflow-wrap:anywhere;word-break:break-word;"></td>
<td style="border-bottom:1px solid #eee;padding:6px 4px;vertical-align:top;text-align:left;overflow-wrap:anywhere;word-break:break-word;"><div style="margin:0 0 4px 0;padding:3px 4px;border-left:3px solid #cbd5e1;background:#fff;border-radius:3px;"><div style="line-height:1.25;"><div style="font-weight:600;">GE1026</div><div>紀錄片賞析與創作</div><div style="display:inline-block;margin-top:2px;padding:1px 4px;background:#f3f4f6;border-radius:3px;font-size:11px;">WnW5W6</div></div></div></td>
<td style="border-bottom:1px solid #eee;padding:6px 4px;vertical-align:top;text-align:left;overflow-wrap:anywhere;word-break:break-word;"></td>
<td style="border-bottom:1px solid #eee;padding:6px 4px;vertical-align:top;text-align:left;overflow-wrap:anywhere;word-break:break-word;"></td>
<td style="border-bottom:1px solid #eee;padding:6px 4px;vertical-align:top;text-align:left;overflow-wrap:anywhere;word-break:break-word;"></td>
<td style="border-bottom:1px solid #eee;padding:6px 4px;vertical-align:top;text-align:left;overflow-wrap:anywhere;word-break:break-word;"></td>
</tr>
<tr style="background:#f7f7f7;">
<td style="border-bottom:1px solid #eee;padding:6px 4px;vertical-align:top;text-align:center;white-space:nowrap;font-weight:600;">5</td>
<td style="border-bottom:1px solid #eee;padding:6px 4px;vertical-align:top;text-align:center;white-space:nowrap;">13:20-14:10</td>
<td style="border-bottom:1px solid #eee;padding:6px 4px;vertical-align:top;text-align:left;overflow-wrap:anywhere;word-break:break-word;"><div style="margin:0 0 4px 0;padding:3px 4px;border-left:3px solid #cbd5e1;background:#fff;border-radius:3px;"><div style="line-height:1.25;"><div style="font-weight:600;">PE1120</div><div>重量訓練</div><div style="display:inline-block;margin-top:2px;padding:1px 4px;background:#f3f4f6;border-radius:3px;font-size:11px;">M5M6</div></div></div></td>
<td style="border-bottom:1px solid #eee;padding:6px 4px;vertical-align:top;text-align:left;overflow-wrap:anywhere;word-break:break-word;"><div style="margin:0 0 4px 0;padding:3px 4px;border-left:3px solid #cbd5e1;background:#fff;border-radius:3px;"><div style="line-height:1.25;"><div style="font-weight:600;">EE3060</div><div>機率</div><div style="display:inline-block;margin-top:2px;padding:1px 4px;background:#f3f4f6;border-radius:3px;font-size:11px;">T5T6R5R6</div></div></div></td>
<td style="border-bottom:1px solid #eee;padding:6px 4px;vertical-align:top;text-align:left;overflow-wrap:anywhere;word-break:break-word;"><div style="margin:0 0 4px 0;padding:3px 4px;border-left:3px solid #cbd5e1;background:#fff;border-radius:3px;"><div style="line-height:1.25;"><div style="font-weight:600;">GE1026</div><div>紀錄片賞析與創作</div><div style="display:inline-block;margin-top:2px;padding:1px 4px;background:#f3f4f6;border-radius:3px;font-size:11px;">WnW5W6</div></div></div></td>
<td style="border-bottom:1px solid #eee;padding:6px 4px;vertical-align:top;text-align:left;overflow-wrap:anywhere;word-break:break-word;"><div style="margin:0 0 4px 0;padding:3px 4px;border-left:3px solid #cbd5e1;background:#fff;border-radius:3px;"><div style="line-height:1.25;"><div style="font-weight:600;">EE3060</div><div>機率</div><div style="display:inline-block;margin-top:2px;padding:1px 4px;background:#f3f4f6;border-radius:3px;font-size:11px;">T5T6R5R6</div></div></div></td>
<td style="border-bottom:1px solid #eee;padding:6px 4px;vertical-align:top;text-align:left;overflow-wrap:anywhere;word-break:break-word;"></td>
<td style="border-bottom:1px solid #eee;padding:6px 4px;vertical-align:top;text-align:left;overflow-wrap:anywhere;word-break:break-word;"></td>
<td style="border-bottom:1px solid #eee;padding:6px 4px;vertical-align:top;text-align:left;overflow-wrap:anywhere;word-break:break-word;"></td>
</tr>
<tr style="background:#fff;">
<td style="border-bottom:1px solid #eee;padding:6px 4px;vertical-align:top;text-align:center;white-space:nowrap;font-weight:600;">6</td>
<td style="border-bottom:1px solid #eee;padding:6px 4px;vertical-align:top;text-align:center;white-space:nowrap;">14:20-15:10</td>
<td style="border-bottom:1px solid #eee;padding:6px 4px;vertical-align:top;text-align:left;overflow-wrap:anywhere;word-break:break-word;"><div style="margin:0 0 4px 0;padding:3px 4px;border-left:3px solid #cbd5e1;background:#fff;border-radius:3px;"><div style="line-height:1.25;"><div style="font-weight:600;">PE1120</div><div>重量訓練</div><div style="display:inline-block;margin-top:2px;padding:1px 4px;background:#f3f4f6;border-radius:3px;font-size:11px;">M5M6</div></div></div></td>
<td style="border-bottom:1px solid #eee;padding:6px 4px;vertical-align:top;text-align:left;overflow-wrap:anywhere;word-break:break-word;"><div style="margin:0 0 4px 0;padding:3px 4px;border-left:3px solid #cbd5e1;background:#fff;border-radius:3px;"><div style="line-height:1.25;"><div style="font-weight:600;">EE3060</div><div>機率</div><div style="display:inline-block;margin-top:2px;padding:1px 4px;background:#f3f4f6;border-radius:3px;font-size:11px;">T5T6R5R6</div></div></div></td>
<td style="border-bottom:1px solid #eee;padding:6px 4px;vertical-align:top;text-align:left;overflow-wrap:anywhere;word-break:break-word;"><div style="margin:0 0 4px 0;padding:3px 4px;border-left:3px solid #cbd5e1;background:#fff;border-radius:3px;"><div style="line-height:1.25;"><div style="font-weight:600;">GE1026</div><div>紀錄片賞析與創作</div><div style="display:inline-block;margin-top:2px;padding:1px 4px;background:#f3f4f6;border-radius:3px;font-size:11px;">WnW5W6</div></div></div></td>
<td style="border-bottom:1px solid #eee;padding:6px 4px;vertical-align:top;text-align:left;overflow-wrap:anywhere;word-break:break-word;"><div style="margin:0 0 4px 0;padding:3px 4px;border-left:3px solid #cbd5e1;background:#fff;border-radius:3px;"><div style="line-height:1.25;"><div style="font-weight:600;">EE3060</div><div>機率</div><div style="display:inline-block;margin-top:2px;padding:1px 4px;background:#f3f4f6;border-radius:3px;font-size:11px;">T5T6R5R6</div></div></div></td>
<td style="border-bottom:1px solid #eee;padding:6px 4px;vertical-align:top;text-align:left;overflow-wrap:anywhere;word-break:break-word;"></td>
<td style="border-bottom:1px solid #eee;padding:6px 4px;vertical-align:top;text-align:left;overflow-wrap:anywhere;word-break:break-word;"></td>
<td style="border-bottom:1px solid #eee;padding:6px 4px;vertical-align:top;text-align:left;overflow-wrap:anywhere;word-break:break-word;"></td>
</tr>
<tr style="background:#f7f7f7;">
<td style="border-bottom:1px solid #eee;padding:6px 4px;vertical-align:top;text-align:center;white-space:nowrap;font-weight:600;">7</td>
<td style="border-bottom:1px solid #eee;padding:6px 4px;vertical-align:top;text-align:center;white-space:nowrap;">15:30-16:20</td>
<td style="border-bottom:1px solid #eee;padding:6px 4px;vertical-align:top;text-align:left;overflow-wrap:anywhere;word-break:break-word;"></td>
<td style="border-bottom:1px solid #eee;padding:6px 4px;vertical-align:top;text-align:left;overflow-wrap:anywhere;word-break:break-word;"></td>
<td style="border-bottom:1px solid #eee;padding:6px 4px;vertical-align:top;text-align:left;overflow-wrap:anywhere;word-break:break-word;"><div style="margin:0 0 4px 0;padding:3px 4px;border-left:3px solid #cbd5e1;background:#fff;border-radius:3px;"><div style="line-height:1.25;"><div style="font-weight:600;">EE2405</div><div>嵌入式系統與實驗</div><div style="display:inline-block;margin-top:2px;padding:1px 4px;background:#f3f4f6;border-radius:3px;font-size:11px;">W7W8W9WaWb</div></div></div></td>
<td style="border-bottom:1px solid #eee;padding:6px 4px;vertical-align:top;text-align:left;overflow-wrap:anywhere;word-break:break-word;"></td>
<td style="border-bottom:1px solid #eee;padding:6px 4px;vertical-align:top;text-align:left;overflow-wrap:anywhere;word-break:break-word;"></td>
<td style="border-bottom:1px solid #eee;padding:6px 4px;vertical-align:top;text-align:left;overflow-wrap:anywhere;word-break:break-word;"></td>
<td style="border-bottom:1px solid #eee;padding:6px 4px;vertical-align:top;text-align:left;overflow-wrap:anywhere;word-break:break-word;"></td>
</tr>
<tr style="background:#fff;">
<td style="border-bottom:1px solid #eee;padding:6px 4px;vertical-align:top;text-align:center;white-space:nowrap;font-weight:600;">8</td>
<td style="border-bottom:1px solid #eee;padding:6px 4px;vertical-align:top;text-align:center;white-space:nowrap;">16:30-17:20</td>
<td style="border-bottom:1px solid #eee;padding:6px 4px;vertical-align:top;text-align:left;overflow-wrap:anywhere;word-break:break-word;"></td>
<td style="border-bottom:1px solid #eee;padding:6px 4px;vertical-align:top;text-align:left;overflow-wrap:anywhere;word-break:break-word;"></td>
<td style="border-bottom:1px solid #eee;padding:6px 4px;vertical-align:top;text-align:left;overflow-wrap:anywhere;word-break:break-word;"><div style="margin:0 0 4px 0;padding:3px 4px;border-left:3px solid #cbd5e1;background:#fff;border-radius:3px;"><div style="line-height:1.25;"><div style="font-weight:600;">EE2405</div><div>嵌入式系統與實驗</div><div style="display:inline-block;margin-top:2px;padding:1px 4px;background:#f3f4f6;border-radius:3px;font-size:11px;">W7W8W9WaWb</div></div></div></td>
<td style="border-bottom:1px solid #eee;padding:6px 4px;vertical-align:top;text-align:left;overflow-wrap:anywhere;word-break:break-word;"></td>
<td style="border-bottom:1px solid #eee;padding:6px 4px;vertical-align:top;text-align:left;overflow-wrap:anywhere;word-break:break-word;"></td>
<td style="border-bottom:1px solid #eee;padding:6px 4px;vertical-align:top;text-align:left;overflow-wrap:anywhere;word-break:break-word;"><div style="margin:0 0 4px 0;padding:3px 4px;border-left:3px solid #cbd5e1;background:#fff;border-radius:3px;"><div style="line-height:1.25;"><div style="font-weight:600;">EE3900</div><div>實作專題一</div><div style="display:inline-block;margin-top:2px;padding:1px 4px;background:#f3f4f6;border-radius:3px;font-size:11px;">S8</div></div></div></td>
<td style="border-bottom:1px solid #eee;padding:6px 4px;vertical-align:top;text-align:left;overflow-wrap:anywhere;word-break:break-word;"></td>
</tr>
<tr style="background:#f7f7f7;">
<td style="border-bottom:1px solid #eee;padding:6px 4px;vertical-align:top;text-align:center;white-space:nowrap;font-weight:600;">9</td>
<td style="border-bottom:1px solid #eee;padding:6px 4px;vertical-align:top;text-align:center;white-space:nowrap;">17:30-18:20</td>
<td style="border-bottom:1px solid #eee;padding:6px 4px;vertical-align:top;text-align:left;overflow-wrap:anywhere;word-break:break-word;"></td>
<td style="border-bottom:1px solid #eee;padding:6px 4px;vertical-align:top;text-align:left;overflow-wrap:anywhere;word-break:break-word;"></td>
<td style="border-bottom:1px solid #eee;padding:6px 4px;vertical-align:top;text-align:left;overflow-wrap:anywhere;word-break:break-word;"><div style="margin:0 0 4px 0;padding:3px 4px;border-left:3px solid #cbd5e1;background:#fff;border-radius:3px;"><div style="line-height:1.25;"><div style="font-weight:600;">EE2405</div><div>嵌入式系統與實驗</div><div style="display:inline-block;margin-top:2px;padding:1px 4px;background:#f3f4f6;border-radius:3px;font-size:11px;">W7W8W9WaWb</div></div></div></td>
<td style="border-bottom:1px solid #eee;padding:6px 4px;vertical-align:top;text-align:left;overflow-wrap:anywhere;word-break:break-word;"></td>
<td style="border-bottom:1px solid #eee;padding:6px 4px;vertical-align:top;text-align:left;overflow-wrap:anywhere;word-break:break-word;"></td>
<td style="border-bottom:1px solid #eee;padding:6px 4px;vertical-align:top;text-align:left;overflow-wrap:anywhere;word-break:break-word;"></td>
<td style="border-bottom:1px solid #eee;padding:6px 4px;vertical-align:top;text-align:left;overflow-wrap:anywhere;word-break:break-word;"></td>
</tr>
<tr style="background:#fff;">
<td style="border-bottom:1px solid #eee;padding:6px 4px;vertical-align:top;text-align:center;white-space:nowrap;font-weight:600;">a</td>
<td style="border-bottom:1px solid #eee;padding:6px 4px;vertical-align:top;text-align:center;white-space:nowrap;">18:30-19:20</td>
<td style="border-bottom:1px solid #eee;padding:6px 4px;vertical-align:top;text-align:left;overflow-wrap:anywhere;word-break:break-word;"></td>
<td style="border-bottom:1px solid #eee;padding:6px 4px;vertical-align:top;text-align:left;overflow-wrap:anywhere;word-break:break-word;"><div style="margin:0 0 4px 0;padding:3px 4px;border-left:3px solid #cbd5e1;background:#fff;border-radius:3px;"><div style="line-height:1.25;"><div style="font-weight:600;">GE1024</div><div>產業創新實作</div><div style="display:inline-block;margin-top:2px;padding:1px 4px;background:#f3f4f6;border-radius:3px;font-size:11px;">TaTb</div></div></div></td>
<td style="border-bottom:1px solid #eee;padding:6px 4px;vertical-align:top;text-align:left;overflow-wrap:anywhere;word-break:break-word;"><div style="margin:0 0 4px 0;padding:3px 4px;border-left:3px solid #cbd5e1;background:#fff;border-radius:3px;"><div style="line-height:1.25;"><div style="font-weight:600;">EE2405</div><div>嵌入式系統與實驗</div><div style="display:inline-block;margin-top:2px;padding:1px 4px;background:#f3f4f6;border-radius:3px;font-size:11px;">W7W8W9WaWb</div></div></div></td>
<td style="border-bottom:1px solid #eee;padding:6px 4px;vertical-align:top;text-align:left;overflow-wrap:anywhere;word-break:break-word;"></td>
<td style="border-bottom:1px solid #eee;padding:6px 4px;vertical-align:top;text-align:left;overflow-wrap:anywhere;word-break:break-word;"></td>
<td style="border-bottom:1px solid #eee;padding:6px 4px;vertical-align:top;text-align:left;overflow-wrap:anywhere;word-break:break-word;"></td>
<td style="border-bottom:1px solid #eee;padding:6px 4px;vertical-align:top;text-align:left;overflow-wrap:anywhere;word-break:break-word;"></td>
</tr>
<tr style="background:#f7f7f7;">
<td style="border-bottom:1px solid #eee;padding:6px 4px;vertical-align:top;text-align:center;white-space:nowrap;font-weight:600;">b</td>
<td style="border-bottom:1px solid #eee;padding:6px 4px;vertical-align:top;text-align:center;white-space:nowrap;">19:30-20:20</td>
<td style="border-bottom:1px solid #eee;padding:6px 4px;vertical-align:top;text-align:left;overflow-wrap:anywhere;word-break:break-word;"></td>
<td style="border-bottom:1px solid #eee;padding:6px 4px;vertical-align:top;text-align:left;overflow-wrap:anywhere;word-break:break-word;"><div style="margin:0 0 4px 0;padding:3px 4px;border-left:3px solid #cbd5e1;background:#fff;border-radius:3px;"><div style="line-height:1.25;"><div style="font-weight:600;">GE1024</div><div>產業創新實作</div><div style="display:inline-block;margin-top:2px;padding:1px 4px;background:#f3f4f6;border-radius:3px;font-size:11px;">TaTb</div></div></div></td>
<td style="border-bottom:1px solid #eee;padding:6px 4px;vertical-align:top;text-align:left;overflow-wrap:anywhere;word-break:break-word;"><div style="margin:0 0 4px 0;padding:3px 4px;border-left:3px solid #cbd5e1;background:#fff;border-radius:3px;"><div style="line-height:1.25;"><div style="font-weight:600;">EE2405</div><div>嵌入式系統與實驗</div><div style="display:inline-block;margin-top:2px;padding:1px 4px;background:#f3f4f6;border-radius:3px;font-size:11px;">W7W8W9WaWb</div></div></div></td>
<td style="border-bottom:1px solid #eee;padding:6px 4px;vertical-align:top;text-align:left;overflow-wrap:anywhere;word-break:break-word;"></td>
<td style="border-bottom:1px solid #eee;padding:6px 4px;vertical-align:top;text-align:left;overflow-wrap:anywhere;word-break:break-word;"></td>
<td style="border-bottom:1px solid #eee;padding:6px 4px;vertical-align:top;text-align:left;overflow-wrap:anywhere;word-break:break-word;"></td>
<td style="border-bottom:1px solid #eee;padding:6px 4px;vertical-align:top;text-align:left;overflow-wrap:anywhere;word-break:break-word;"></td>
</tr>
<tr style="background:#fff;">
<td style="border-bottom:1px solid #eee;padding:6px 4px;vertical-align:top;text-align:center;white-space:nowrap;font-weight:600;">c</td>
<td style="border-bottom:1px solid #eee;padding:6px 4px;vertical-align:top;text-align:center;white-space:nowrap;">20:30-21:20</td>
<td style="border-bottom:1px solid #eee;padding:6px 4px;vertical-align:top;text-align:left;overflow-wrap:anywhere;word-break:break-word;"></td>
<td style="border-bottom:1px solid #eee;padding:6px 4px;vertical-align:top;text-align:left;overflow-wrap:anywhere;word-break:break-word;"></td>
<td style="border-bottom:1px solid #eee;padding:6px 4px;vertical-align:top;text-align:left;overflow-wrap:anywhere;word-break:break-word;"></td>
<td style="border-bottom:1px solid #eee;padding:6px 4px;vertical-align:top;text-align:left;overflow-wrap:anywhere;word-break:break-word;"></td>
<td style="border-bottom:1px solid #eee;padding:6px 4px;vertical-align:top;text-align:left;overflow-wrap:anywhere;word-break:break-word;"></td>
<td style="border-bottom:1px solid #eee;padding:6px 4px;vertical-align:top;text-align:left;overflow-wrap:anywhere;word-break:break-word;"></td>
<td style="border-bottom:1px solid #eee;padding:6px 4px;vertical-align:top;text-align:left;overflow-wrap:anywhere;word-break:break-word;"></td>
</tr>
<tr style="background:#f7f7f7;">
<td style="border-bottom:1px solid #eee;padding:6px 4px;vertical-align:top;text-align:center;white-space:nowrap;font-weight:600;">d</td>
<td style="border-bottom:1px solid #eee;padding:6px 4px;vertical-align:top;text-align:center;white-space:nowrap;">21:30-22:20</td>
<td style="border-bottom:1px solid #eee;padding:6px 4px;vertical-align:top;text-align:left;overflow-wrap:anywhere;word-break:break-word;"></td>
<td style="border-bottom:1px solid #eee;padding:6px 4px;vertical-align:top;text-align:left;overflow-wrap:anywhere;word-break:break-word;"></td>
<td style="border-bottom:1px solid #eee;padding:6px 4px;vertical-align:top;text-align:left;overflow-wrap:anywhere;word-break:break-word;"></td>
<td style="border-bottom:1px solid #eee;padding:6px 4px;vertical-align:top;text-align:left;overflow-wrap:anywhere;word-break:break-word;"></td>
<td style="border-bottom:1px solid #eee;padding:6px 4px;vertical-align:top;text-align:left;overflow-wrap:anywhere;word-break:break-word;"></td>
<td style="border-bottom:1px solid #eee;padding:6px 4px;vertical-align:top;text-align:left;overflow-wrap:anywhere;word-break:break-word;"></td>
<td style="border-bottom:1px solid #eee;padding:6px 4px;vertical-align:top;text-align:left;overflow-wrap:anywhere;word-break:break-word;"></td>
</tr>
</tbody>
</table>

#### 學分狀態
- 目前總學分：**20**
- 正常學分範圍：**16-25**
- 低修：**False**
- 超修：**False**

#### 重要提醒
- 修課中的課只是在規劃模式假設會通過；如果沒有通過，畢業進度和推薦要重算。
- 這不是正式畢業審查，最後仍要以系辦確認為準。

#### 時間代碼說明
##### 星期代碼
| 代碼 | 星期 | English |
|---|---|---|
| `M` | 星期一 | Monday |
| `T` | 星期二 | Tuesday |
| `W` | 星期三 | Wednesday |
| `R` | 星期四 | Thursday |
| `F` | 星期五 | Friday |
| `S` | 星期六 | Saturday |
| `U` | 星期日 | Sunday |

##### 節次時間
| 節次 | 時間 |
|---|---|
| `1` | 08:00-08:50 |
| `2` | 09:00-09:50 |
| `3` | 10:10-11:00 |
| `4` | 11:10-12:00 |
| `n` | 12:10-13:00 |
| `5` | 13:20-14:10 |
| `6` | 14:20-15:10 |
| `7` | 15:30-16:20 |
| `8` | 16:30-17:20 |
| `9` | 17:30-18:20 |
| `a` | 18:30-19:20 |
| `b` | 19:30-20:20 |
| `c` | 20:30-21:20 |
| `d` | 21:30-22:20 |

例：`T3T4R3R4` 代表星期二第 3、4 節，和星期四第 3、4 節。

### Calendar Export
已自動匯出行事曆檔：`/homes/jupyter_student/jupyter-112000271/Test_2/final_schedule.ics`

In [4]:
if recommendation is not None:
    display(Markdown("已保存最新確認或推薦課表為 `recommendation`。"))
else:
    display(Markdown("目前尚未產生課表；請重新執行 Demo 2 並輸入排課需求。"))


已保存最新確認或推薦課表為 `recommendation`。

## Extension Demo: Evaluation Report

這個區塊產生簡單評估表，用於報告中說明 intent parsing、衝堂檢查、OCR 課號抽取與 calendar export 的測試結果。

In [5]:
display(Markdown("### Evaluation Table"))
evaluation = run_evaluation_report(
    student_path=student_path,
    target_path=target_path,
    output_csv_path="evaluation_results.csv",
)
display(Markdown(evaluation["markdown_table"]))
display(Markdown(f"CSV: `{evaluation['csv_path']}`"))


### Evaluation Table

| Test category | Test input | Expected behavior | Actual result | Pass/Fail | Notes |
| --- | --- | --- | --- | --- | --- |
| Intent parsing | 我要修20學分 | Parse a target-credit schedule request | action=recommend_schedule, operation=none, courses=[] | Pass | Intent parsing only creates structured intent; deterministic tools still execute the result. |
| Intent parsing | 我不想上實驗課 | Persistently exclude lab/experiment courses | action=modify_schedule, operation=remove_category, courses=[] | Pass | Intent parsing only creates structured intent; deterministic tools still execute the result. |
| Intent parsing | 但我想上固態電子實驗 | Recognize a specific lab course override/add request | action=modify_schedule, operation=add_course, courses=['固態電子實驗'] | Pass | Intent parsing only creates structured intent; deterministic tools still execute the result. |
| Intent parsing | 我不要星期五的課 | Recognize Friday as an avoided day | action=modify_schedule, operation=remove_day, courses=[] | Pass | Intent parsing only creates structured intent; deterministic tools still execute the result. |
| Intent parsing | 我不要上早上八點到十點的課 | Recognize first and second periods as avoided time slots | action=modify_schedule, operation=remove_day, courses=[] | Pass | Intent parsing only creates structured intent; deterministic tools still execute the result. |
| Conflict checking | EE2020 T3T4 + TEST1000 T3F4 | Detect overlap at T3 | [{'course_a': 'EE2020 偏微分方程與複變函數', 'course_b': 'TEST1000 Test Course', 'overlap_slots': ['T3']}] | Pass | Uses deterministic schedule_checker.parse_time_slots/check_plan_conflicts. |
| OCR extraction | EE2020 偏微分方程與複變函數 T3T4R3R4 | Extract and normalize at least one course code | codes=['EE2020'], matched_target=0 | Pass | Manual OCR fallback is used; no schedule is modified. |
| OCR extraction | 11420EE 214001 電磁學 M3M4W2 | Extract and normalize at least one course code | codes=['EE2140'], matched_target=0 | Pass | Manual OCR fallback is used; no schedule is modified. |
| OCR extraction | EE3060 機率 T5T6R5R6 | Extract and normalize at least one course code | codes=['EE3060'], matched_target=0 | Pass | Manual OCR fallback is used; no schedule is modified. |
| Calendar export | Fake EE2020 timed course + TBA course | Generate .ics and skip TBA course | ics=/homes/jupyter_student/jupyter-112000271/Test_2/evaluation_calendar_test.ics, exported=3, skipped=0 | Fail | Uses only Python standard library for ICS generation. |

CSV: `/homes/jupyter_student/jupyter-112000271/Test_2/evaluation_results.csv`